# 01 — MuJoCo Playground で四脚（Go1）の歩行を学習する

テスト機 `Go1JoystickFlatTerrain` を Brax PPO で学習し、歩く動画を出すまで。

**準備**: ランタイム → ランタイムのタイプを変更 → **GPU（T4 以上）**。

ロジックは `quadleg_rl/train.py` にあり、このノートブックは呼び出すだけ。コードの編集は Zed で行い、GitHub に push → 下のセルで pull。

In [ ]:
#@title 1. GPU 確認
!nvidia-smi -L

In [ ]:
#@title 2. リポジトリ取得（GitHub に push 済みの URL を入れる。未 push なら左のファイル欄に quadleg_rl/ フォルダをアップロード）
REPO_URL = "https://github.com/<your-name>/quadleg-rl.git"  #@param {type:"string"}
import os, sys
if REPO_URL and "<your-name>" not in REPO_URL:
    if os.path.isdir("/content/quadleg-rl"):
        !cd /content/quadleg-rl && git pull
    else:
        !git clone $REPO_URL /content/quadleg-rl
    %cd /content/quadleg-rl
else:
    print("REPO_URL 未設定。/content に quadleg_rl/ を置いてください。")
    %cd /content
sys.path.insert(0, os.getcwd())

In [ ]:
#@title 3. インストール（初回 2〜3 分。ランタイム再起動は不要）
%pip install -q -U "jax[cuda12]" playground mediapy
import jax; print("JAX backend:", jax.default_backend(), jax.devices())

In [ ]:
#@title 4. Google Drive をマウント（チェックポイント・動画の保存先）
from google.colab import drive
drive.mount("/content/drive")
LOGDIR = "/content/drive/MyDrive/quadleg-rl/logs"
import os; os.makedirs(LOGDIR, exist_ok=True); print(LOGDIR)

In [ ]:
#@title 5. 学習（T4: 2,000 万ステップで 20〜40 分。まず短く回して流れを確認）
ENV_NAME = "Go1JoystickFlatTerrain"  #@param ["Go1JoystickFlatTerrain", "Go1JoystickRoughTerrain", "Go1Getup", "SpotFlatTerrainJoystick", "BarkourJoystick"]
NUM_TIMESTEPS = 20_000_000  #@param {type:"integer"}
NUM_EVALS = 10  #@param {type:"integer"}

from quadleg_rl import train
result = train.train(ENV_NAME, num_timesteps=NUM_TIMESTEPS, num_evals=NUM_EVALS, logdir=LOGDIR)
train.plot_history(result)

In [ ]:
#@title 6. 歩行動画（前進 0.5 m/s の指令を固定）
import mediapy as media
path = train.render_video(result, result.logdir / "rollout_forward.mp4", command=(0.5, 0.0, 0.0))
media.show_video(media.read_video(path), fps=25)

In [ ]:
#@title 7. 旋回動画（ヨー 1 rad/s）
path = train.render_video(result, result.logdir / "rollout_turn.mp4", command=(0.0, 0.0, 1.0))
media.show_video(media.read_video(path), fps=25)

## 次のステップ

- `NUM_TIMESTEPS` を 1 億にすると公式設定と同じ（T4 では数時間。Colab Pro の L4/A100 推奨）。
- 切断されたら、セル 5 で `train.train(..., restore_checkpoint_path=f"{LOGDIR}/<run>/checkpoints")` で再開。
- 自分の脚（XL-320/XL330 四脚）に差し替えるには `quadleg_rl/` に MJCF と環境クラスを追加する（02 ノートブック予定）。